# 1 — Create `df_log` from the log files

**This notebook makes the dataset. It does not analyse it** — that is notebook 2.

Everything here comes from `log.json` and nothing else: no video, no optic flow. That is what lets
it run over a whole server directory whose sessions were never put through the video pipeline.

It produces exactly **two tables**:

| | one row per | holds |
|---|---|---|
| `df_sessions` | SESSION | who/when/which world/which protocol + **the performance numbers (D, chance, throughput…)** |
| `df_trials` | TRIAL | the trial window, path geometry, the on-screen icons, the outcome, the cluster and the error/conflict labels |

Both are saved to `MAIN_DIR/df_log/`. Notebook 2 loads them and never touches the server again.

**The trial logic is not written here.** `task_*/build_trials.py` already defines a trial correctly —
a **spawn batch**, so a batch that ends without a collection (a reshuffle, the session-end tail) is a
real row — and it carries the `icons` list, the `NORMAL`/`BANISH_WORLD` column, and
`start_frame`/`end_frame`. Those builders are called here directly, and **nothing is read from or
written into the session folders**.

Sections **A** are sanity checks on the data; sections **B** build; section **C** verifies and saves.

## Config ← YOU SET THIS

In [ ]:
MAIN_DIR = '/mnt/server/data'     # the directory holding one folder per animal
VIEW_SCALES = {}                  # {world signature: scale} for any world without a known scale
PIPELINE_DIR = None               # None = locate session_pipeline/ automatically
# =============================================================================

import sys, json, importlib
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

MAIN_DIR = Path(MAIN_DIR).expanduser()
cands = ([Path(PIPELINE_DIR)] if PIPELINE_DIR else []) + [
    Path.cwd().parent, Path.cwd(), Path.cwd().parent / 'session_pipeline']
PIPE = next((c.resolve() for c in cands if (c / 'common' / 'session_index.py').exists()), None)
if PIPE is None:
    raise FileNotFoundError('set PIPELINE_DIR. Tried: ' + ', '.join(str(c) for c in cands))
sys.path.insert(0, str(PIPE / 'common')); sys.path.insert(0, str(PIPE / 'performance'))

import session_index as sidx, perf_from_log as pfl, build_log_df as bl
# RELOAD, don't just import: Python caches a module, so after a `git pull` the kernel would keep
# running the OLD code while re-running this cell looked like it worked.
sidx = importlib.reload(sidx); pfl = importlib.reload(pfl); bl = importlib.reload(bl)

print(f'pipeline : {PIPE}')
print(f'main dir : {MAIN_DIR}')
print(f'builders : {list(bl.TASK_MODULES)}   (others fall back to the generic builder)')

## A1 — SANITY: what is actually on disk?

Deliberately dumb: it lists folders and counts files, **without opening a single log**. If the mount
is missing, the path is mis-typed, or a sync is half-finished, it looks wrong *here* — rather than
showing up later as a mysteriously small dataset.

In [ ]:
ANIMALS = sidx.list_animals(MAIN_DIR)
display(ANIMALS)

for a in ANIMALS.animal:
    print(f'\n--- {a} ---')
    display(sidx.list_sessions(MAIN_DIR / a).head(8))

## A2 — What a log actually contains

One log, opened and shown, so the rest of the notebook is readable: you can see the fields every
column below is derived from.

In [ ]:
_first = next((MAIN_DIR / a).glob('*/log.json'), None) or next(MAIN_DIR.glob('*/*/log.json'))
_L = json.load(open(_first))
print(_first, '\n')
print('top-level keys :', list(_L.keys()), '\n')
print('experiment_data:', _L.get('experiment_data'), '\n')
# collected[]: one row per collection. It carries the reward `multiplier` (for the current world and
# the next), the effect taken, and where.
print('collected[0]   :', (_L.get('collected') or [{}])[0], '\n')
# spawns[]: each has `current` = the icons ON THE BOARD at that spawn. THIS is what the task is read
# from, per trial -- shown in full for the first spawn so the source is visible, not hidden.
print('spawns[0]      :', (_L.get('spawns') or [{}])[0], '\n')
print('worlds[0]      :', (_L.get('worlds') or [{}])[0], '\n')

# where the TASK comes from: the board (current icons) of each of the first trials -> classified task.
# Line this up against the video to confirm it is reading the right thing.
pfl.show_first_trials(_first)

## A3 — SANITY: read every log and check the identities

`discover()` opens each log and reads what it refuses to guess: the **date**
(`experiment_data.datetime`), the **animal** (the ID's digits, falling back to the folder name), and
the **task**. The task is decided from the **first 10 trials' boards** (`pfl.first_trials_task`): each
trial's task is read from that spawn batch's icons, and if the first 10 are one task the session is
that task. A single differing trial-0 board is allowed (the timeout→banishment shaping switch);
anything less consistent is left with its settled guess but flagged `task_stable = False` so it is
**shown** rather than trusted. The whole-log union of effects is kept as `task_union` for comparison.

Everything printed here is a **check**, not a result.

In [ ]:
S = pd.concat([sidx.discover(MAIN_DIR / a, view_scales=VIEW_SCALES) for a in ANIMALS.animal],
               ignore_index=True)
print(f'\n{len(S)} session(s), {S.mouse.nunique()} animal(s): {sorted(S.mouse.dropna().unique())}')

# animal resolved from the FOLDER rather than the log ID?
if (S.mouse_src == 'folder').any():
    n = int((S.mouse_src == 'folder').sum())
    print(f'\n{n} session(s) took the animal from the folder name (the log ID had no number):')
    display(S.loc[S.mouse_src == 'folder', ['session', 'mouse_raw', 'mouse']].head())

# days holding more than one session
dup = S.day.duplicated(keep=False)
if dup.any():
    print('\ndays with more than one session:')
    display(S.loc[dup, ['session', 'day', 'time', 'name']])

display(S[['session', 'mouse', 'day', 'task', 'task_stable', 'world', 'view_scale', 'use', 'note']].head(12))

### A4 — how many sessions of each task?

The number this notebook is for: how many sessions of each protocol, and specifically
**banish_multiplier vs timeout_multiplier**. The task is the **majority of each session's first 10
trials** (a clean trial-0 → banishment switch still counts as the task it settles into). A session
with no clear majority is listed as `task_stable = False` so you can check it against the video with
`pfl.show_first_trials(log_path)`.

(World numbering is deliberately not shown here — the log has no world id, and the invented W-labels
don't match the lab's region-based worlds. `sidx.task_protocol(S)` still prints them if ever needed.)

In [ ]:
print('TASKS in this directory:')
print(S.task.value_counts().to_string())

print(f"\nbanish_multiplier : {int((S.task == 'banish_multiplier').sum())}")
print(f"timeout_multiplier: {int((S.task == 'timeout_multiplier').sum())}")

# sessions with no clear task majority in their first 10 trials -> check against the video
_uns = S[~S.task_stable.fillna(False)]
if len(_uns):
    print(f'\n{len(_uns)} session(s) with no clear task majority in the first 10 trials:')
    display(_uns[['session', 'task', 'task_seq']])
else:
    print('\nevery session has a clear task majority in its first 10 trials.')

## B1 — BUILD the two tables

One call. For each session: the task's `build_trials` → `cluster_paths` → `label_trials` (all
log-only), then the performance numbers merged into the session row.

**Every discovered session becomes a row.** One that cannot be scored keeps its identity, gets `NaN`
performance and a reason in `perf_error`, and stays in the table — filtering is notebook 2's
explicit choice, not a deletion baked in here. Days holding two sessions are **flagged**
(`is_dup_day`, `keep_of_day`), not dropped.

In [ ]:
df_sessions, df_trials = bl.build_all(MAIN_DIR, view_scales=VIEW_SCALES)

## Column glossary — what each column means, how it is computed, and where in the code

Most columns are **not copied from the log**; they are derived here. This cell explains the ones that
confuse, and each group names the **file → function** that produces it, so you can open the source and
check the definition yourself.

---
### `df_sessions` — one row per session

**Task / identity (derived, not in the log)** — `performance/build_log_df.py` → `session_row()`; the
classification itself is `common/perf_from_log.py` → `classify_task()` / `first_trials_task()`:
- **`task_description`** — a one-line plain-English description of the protocol, looked up from the
  classified `task` (`common/perf_from_log.py` → `TASK_DESCRIPTION`). Not measured.
- **`has_multiplier`** — `True` if **any** collected event carries a non-null `multiplier` field
  (`common/perf_from_log.py` → `log_has_multiplier()`). This is what separates the two timeout tasks:
  the CURRENT `timeout_multiplier` logs a streak multiplier, the OLD `timeout_double` never did.

**Session facts** — `performance/build_log_df.py` → `session_row()`:
- **`n_trials_total`** — number of trials (= spawn batches) in this session's trial table, **including**
  reshuffle and incomplete batches.
- **`n_analyzable`** — trials with `analyze == True` (a real collection, not degenerate). ≤ `n_trials_total`.
- **`elapsed_min`** *(renamed from `wall_min`)* — real clock minutes from the first scored trial to the
  last. It **splits into `active_min + freeze_min`**. Computed in `common/perf_from_log.py` →
  `score_log()`; for unscorable sessions it falls back to the coord-stream span in `session_row()`.
- **`active_min`** — `elapsed_min` minus the timeout freeze: the time the animal could actually move.
  Equals `elapsed_min` for the banish task (no freeze). *(`score_log()`)*
- **`freeze_min`** — total minutes lost to the ~14 s post-timeout blackout (timeout task only; 0 for
  banish). *(`score_log()`)*
- **`weight_g` / `weight_baseline_g` / `weight_fraction`** — the animal's weight that day, its
  free-feeding baseline, and the ratio, read from the log's `experiment_data` (these *are* in the log).
  *(`session_row()`)*
- **`n_single_reward`** (and `n_banish`, `n_timeout`, …) — one `n_<effect>` per outcome type, straight
  from `outcome.value_counts()`. *(`session_row()`)*
- **`perf_error`** — empty string if scoring succeeded; otherwise the exception message. A session that
  can't be scored keeps its identity and gets `NaN` performance + this reason, rather than vanishing.
  *(`session_row()`)*
- **`switch_ms` / `n_before_switch`** — the **PROTOCOL switch** (board changed task at trial 0, e.g.
  timeout→banishment): the switch time and how many trials preceded it. Detected by
  `common/perf_from_log.py` → `protocol_switch()`, applied in `score_log()`. Performance is scored only
  on the part AFTER the switch. **This is the board changing, NOT the camera moving** (that's `camera_*`).
- **`camera_stable` / `camera_move_frame` / `camera_move_ms`** — from the camera check, a
  **video-pipeline** output read in `common/session_index.py` → `discover()` (from `session.json` +
  `opticflow/camera_check.json`) and copied through `session_row()`: `True` steady · `False` moved ·
  **`None` not checked yet**. Frame/ms are `NaN` unless the camera moved.

**Performance — `common/perf_from_log.py` → `score_log()` (log only, no video), merged into the row by
`session_row()`:**

*Counts & accuracy*
- **`n_trials`** — scorable trials only (positive+negative collections; excludes reshuffle / incomplete /
  escape / degenerate, and the pre-switch lead-in). Differs from `n_trials_total`.
- **`pos` / `neg`** — collections of a positive (reward) / negative (timeout or banish) icon.
- **`drops`** — total reward units earned: the streak **multiplier** for each `single_reward` where one
  is logged, else the fixed pay (single=1, double=2, timeout/banish/unbanish=0; `FIXED_DROPS`).
- **`acc`** — observed accuracy = `pos / (pos+neg)` = P(collected a positive icon | collected something).

*Throughput — how often / how much he scored per minute (`score_log()`):*
- **`coll_per_min`** — positive collections per ELAPSED minute = `pos / elapsed_min`.
- **`drops_per_min`** — reward drops per ELAPSED minute = `drops / elapsed_min` (drops ≥ collections
  because a streak pays the multiplier).
- **`coll_per_active_min`** — positive collections per ACTIVE minute = `pos / active_min` (elapsed with
  the timeout freeze removed). Equals `coll_per_min` for the banish task, where there is no freeze; the
  gap between the two on the timeout task is the throughput cost of the freeze penalty.

*The chance baseline — why raw `acc` isn't comparable across sessions.* An animal that can't tell the
icons apart still scores above 0.5 because reward icons are on screen more often. So we weight chance by
**what was actually visible**, at two memory assumptions (real memory is between — D is a range):
- **`vis_pos` / `vis_neg`** — ZERO-memory: time-weighted fraction of the trial a positive / negative icon
  was inside the viewport, averaged over scorable trials.
- **`chance`** — ZERO-memory baseline = `vis_pos / (vis_pos + vis_neg)`.
- **`mem_pos` / `mem_neg`** — PERFECT-memory: fraction of trials in which a positive / negative icon was
  **ever** on screen (seen at least once).
- **`chance_mem`** — PERFECT-memory baseline = `mem_pos / (mem_pos + mem_neg)`.

*The discrimination score* **D = (acc − chance) / (1 − chance)** — the fraction of the headroom above
chance that he captured: **0 = at chance, 1 = perfect hazard avoidance, <0 = worse than chance.** `NaN`
if `chance ≥ 1` (no headroom). Formula in the local `toD()` inside `score_log()`.
- **`D` / `D_lo` / `D_hi` / `p`** — D at the ZERO-memory chance, its Wilson 95% CI (the CI is on `acc`,
  mapped through the D formula; `_wilson()`), and the binomial p of `acc` vs `chance`.
- **`D_mem` / `p_mem`** — the same D and p at the PERFECT-memory chance.
- **`chance_exo` / `D_exo` / `p_exo`** — an EXOGENOUS cross-check: fraction of trials where the **nearest
  icon at spawn** was positive (fixed by the board before he acts, so his behaviour can't move it), and D
  / p against it.

*The learning criterion — the number to track across days.* On trials where a positive AND a negative
icon were on screen **together**, split by which was nearer at that first joint moment:
- **`n_both`** — both types visible together (= `n_conflict + n_agree`).
- **`n_conflict`** — of those, the negative icon was **nearer** (proximity and value disagree — he must
  override the pull of the closer icon). **`k_conflict`** — how many of those he still took the positive.
- **`conflict_p`** (+ **`conflict_lo` / `conflict_hi`** Wilson CI) — `k_conflict / n_conflict` =
  P(positive | conflict). **This must RISE with training if he is learning to avoid the hazard.**
- **`n_agree` / `agree_p`** — the CONTROL: both visible but the positive icon was nearer (easy). Should
  stay high. If both columns move together, he's merely stopped following proximity — not the same as
  recognising the icon.

*The world-design "opportunity" baseline (Maryam's `active_benefits` / `active_detriments`) —
`common/perf_from_log.py` → `world_opportunity()` + `prob_at_least()`, merged by `score_log()`.* A
baseline built from the WORLD's icon design rather than from what was on screen:
- **`active_benefits` / `active_detriments`** — how many GOOD / BAD icons the world puts on the board,
  from `world['effects']` (benefit set = `single_reward`, `double_reward`, `money`; detriment set =
  `timeout`, `banish`, `debt`). `money` / `debt` are not in the current sessions — they're kept only
  so a future task generation is counted the same way. Read from the log's stored counts where present
  (newer sessions), else counted from the effect names.
- **`good_opportunity_ratio`** — `active_benefits / (active_benefits + active_detriments)`. The
  STOCHASTIC baseline: the hit rate expected if he collected icons in proportion to how many good vs
  bad the world offers, with no discrimination at all (for a 2-good/1-bad world = 0.667). ⚠️ The log's
  own `good_ratio` field is the ODDS (benefits ÷ detriments, e.g. 2.0), **not** this probability — we
  compute the probability ourselves.
- **`p_opportunity`** — the one-sided binomial tail `P(≥ pos positives in pos+neg collections |
  good_opportunity_ratio)` (Maryam's `prob_at_least_y_in_x`). Small = he collects positives *more* often
  than the world's good:bad supply would give by chance.

---
### `df_trials` — one row per trial (a spawn batch)

**Trial status** — `task_banish_multiplier/build_trials.py` → `build()`:
- **`degenerate`** — trial shorter than 0.5 s (a rapid successive / double collection); excluded from analysis.
- **`analyze`** — `True` = ended in a real collection (reward / banish / unbanish) AND not degenerate;
  `False` = reshuffle (board respawned, nothing collected) / incomplete (session-end tail) / degenerate.
- **`builder`** — which trial builder produced the row: the **task name** (validated spawn-batch logic)
  or **`'generic'`** (fallback collection-to-collection builder for a protocol with no dedicated builder).
  Set in `performance/build_log_df.py` → `session_trials()`.

**Outcome valence** — `performance/build_log_df.py` → `session_trials()` (mapped from `outcome` via the
`VALENCE` dict there):
- **`valence`** — `+1` positive (single/double reward) · `−1` negative (timeout/banish) · `0` neutral
  (`unbanish` — an escape is a collection but pays nothing and isn't a hazard hit).
- **`is_positive` / `is_negative`** — `valence == +1` / `== −1`.
- **`drops`** *(per trial)* — the reward units this trial paid (same rule as the session `drops`).

**On-screen board** — `task_banish_multiplier/build_trials.py` → `build()`:
- **`icons`** — the raw list of icons on the board this trial (effect, x, y in world units).
- **`n_icons`** — how many icons were on the board.
- **`n_reward_icons` / `n_banish_icons`** — counts of reward / banish icons among them.
- **`board_benefits` / `board_detriments` / `board_good_ratio`** *(banish_multiplier only)* — the
  per-trial version of the opportunity ratio: benefit / detriment icons on **THIS trial's board** and
  their ratio `board_benefits / (board_benefits + board_detriments)`. **Named `board_*` on purpose** —
  it is a *different* quantity from the session `active_benefits` / `good_opportunity_ratio`, which are
  the WORLD DESIGN; `board_good_ratio` is what was actually on the board that trial, so it varies
  (the shadow-realm escape board has neither → `NaN`). Source: `performance/build_log_df.py` →
  `session_trials()`.

**Path clustering** — `task_banish_multiplier/cluster_paths.py` → `run()` / `cluster()` (four
path-geometry features: efficiency / speed_std / time_in_corner / duration):
- **`cluster`** — KMeans cluster id.
- **`cluster_name`** — `'Direct'` / `'Corner-dwelling'`, named by efficiency rank.
- **`cluster_pca1` / `cluster_pca2`** — 2-D PCA of the feature vector, for the diagnostic scatter only
  (not used in any test).

**Error / conflict labels** — `task_banish_multiplier/label_trials.py` → `run()` (distances in world units):
- **`min_dist_target`** — closest approach to the **collected target** during the trial.
- **`min_dist_banish`** — closest approach to **any banish icon** on the board.
- **`overshoot`** — approached the target within 400 wu, retreated past 700 wu, then returned to collect
  (a terminal overshoot). *(`_overshoot()`)*
- **`is_error_banish`** — `outcome == 'banish'` (the primary, clean error).
- **`is_error`** — `is_error_banish AND analyze`.
- **`is_error_broad`** — `(is_error_banish OR overshoot) AND analyze` — an optional wider error definition
  (overshoot is kept OUT of the primary `is_error` because it conflates true overshoots with long wandering).
- **`is_correct`** — `single_reward AND not overshoot AND analyze`.
- **`is_conflict`** — `cluster_name == 'Corner-dwelling' AND analyze` — the well-powered conflict stratifier.
- **`prev_is_error` / `prev_is_conflict`** — the previous trial's `is_error_banish` / Corner-dwelling
  status (index order), for the post-error / post-conflict tests.


## How "on screen" / the world view is computed

The visibility-weighted `chance` columns (`vis_pos`/`vis_neg`/`chance`, `chance_mem`, and the
conflict criterion) all rest on one question: **was an icon on the mouse's screen at a given moment?**
Here is exactly how that is decided (source: `common/perf_from_log.py` → `score_log()`; the same box
is drawn by `task_banish_multiplier/reconstruct_view.py`).

The game camera **follows the avatar** — `translate(centre); scale(view_scale); translate(-avatar)` —
so the screen shows a box of `SKETCH / view_scale` **world units, centred on the avatar**. `SKETCH` is
the game's canvas, **800 × 600 px**. An icon counts as on screen when its **centre** is inside that box:

```python
SKETCH_W, SKETCH_H = 800, 600                       # the game's sketch (canvas) size in pixels
vw, vh = SKETCH_W / view_scale, SKETCH_H / view_scale   # the viewport, in WORLD units
# icon visible this sample if its CENTRE is inside a box centred on the avatar (ax, ay):
vis = (np.abs(ic['x'] - ax) <= vw/2) & (np.abs(ic['y'] - ay) <= vh/2)
```

**Worked example (JPAS_0168):**

| session | view_scale | viewport = 800/scale × 600/scale | world |
|---|---|---|---|
| JPAS_0168 | 0.35 | **2285.7 × 1714.3 wu** | 2400 × 2400 |

**Three things to keep in mind:**
- **`view_scale` is a property of the WORLD, not in the log**, and must be supplied per session
  (`common/viewport.py`). A wrong scale rescales the whole viewport and shifts every visibility number
  silently. Only the **ratio** `SKETCH / view_scale` matters (the viewport is that many world units across).
- It is **centre-inside**: an icon is counted only once its centre enters the box, so it slightly
  **under-counts** partially-visible icons (icons are ~22 % of the view width). Sprite-overlap would
  raise `chance` a little; left on centre-inside deliberately.
- It is **geometric** frame-membership, not photoreceptor perception — fine for today's green-reward /
  blue-banish icons (both mouse-visible), but it does not model the mouse's near-blindness to red.

The **`good_opportunity_ratio`** baseline does **not** use the viewport at all (it is only the world's
good : bad icon counts), so it is a useful cross-check when the viewport assumptions are in doubt.

## C1 — VERIFY the dataset

Checks that it is right, not that it is interesting. Anything printing `FAIL` needs looking at
before the tables are used.

In [ ]:
def check(name, ok, detail=''):
    print(f'  [{"PASS" if ok else "FAIL"}] {name}' + (f'   {detail}' if detail else ''))

print('DATASET CHECKS')
check('every session has a unique name', df_sessions.session.is_unique)
check('every trial belongs to a listed session',
      set(df_trials.session) <= set(df_sessions.session))

# trial counts agree between the two tables
per = df_trials.groupby('session').size()
agree = all(int(per.get(r.session, 0)) == int(r.n_trials_total) for _, r in df_sessions.iterrows())
check('trials per session match df_sessions.n_trials_total', agree)

# reward drops, computed two independent ways. score_log scores only the part AFTER a protocol
# switch (after_switch=True), so for a switch session its SESSION total excludes the trial-0
# lead-in while the TRIAL table keeps every trial. Show BOTH the all-trials sum and the post-switch
# sum, and pass if EITHER matches the session total -- so a genuine mismatch stands out from the
# benign switch-boundary case, which is labelled by switch_ms / n_before_switch.
_sw = df_sessions.set_index('session').switch_ms if 'switch_ms' in df_sessions else pd.Series(dtype=float)
_g = df_trials.groupby('session')
def _sums(s):
    m = df_trials[df_trials.session == s]
    allsum = float(m.drops.sum())
    post = allsum
    if s in _sw.index and pd.notna(_sw.get(s)) and 'start_ms' in m:
        post = float(m[m.start_ms >= _sw[s]].drops.sum())
    return allsum, post
_d_se = df_sessions.set_index('session').drops
_both = [s for s in _d_se.index if pd.notna(_d_se[s]) and s in set(df_trials.session)]
_rows = []
for s in _both:
    a, po = _sums(s)
    tot = float(_d_se[s])
    if min(abs(a - tot), abs(po - tot)) > 1e-6:      # neither interpretation matches -> real
        _rows.append((s, a, po, tot))
check('reward drops agree (trial sum vs session total)', not _rows, f'{len(_both)} session(s)')
if _rows:
    print(f'    {len(_rows)} session(s) disagree -- neither all-trials nor post-switch sum matches:')
    _bt = pd.DataFrame(_rows, columns=['session', 'trial_sum_all', 'trial_sum_postswitch',
                                       'session_total'])
    _cols = [c for c in ['session', 'task', 'switch_ms', 'n_before_switch'] if c in df_sessions]
    display(_bt.merge(df_sessions[_cols], on='session', how='left').head(20))

check('no session is missing a world', df_sessions.world_sig.notna().all())
n_bad = int((df_sessions.perf_error != '').sum())
check('all sessions scored', n_bad == 0, f'{n_bad} without performance')
if n_bad:
    display(df_sessions.loc[df_sessions.perf_error != '', ['session', 'task', 'perf_error']])

gen = df_trials.builder.eq('generic').sum() if 'builder' in df_trials else 0
if gen:
    print(f'\n  NOTE {gen} trial(s) came from the GENERIC builder (no dedicated one for that '
          f'protocol yet): {sorted(df_trials.loc[df_trials.builder == "generic", "task"].unique())}')
    print('       They have the common columns only -- no spawn-batch trials, no cluster/labels.')

### C2 — Cross-check against a session built by the full pipeline

**This cross-check only makes sense AFTER the tracking (video) pipeline has run on a session.** It
compares the log-only trial table against the `df_trials_clean.pkl` that tracking produces, so it
needs that reference file to exist. On raw server data (log only) there is nothing to compare yet,
and the cell says so rather than doing it silently — that is expected, not a failure.

In [ ]:
_n_checked = 0
for _, r in df_sessions.iterrows():
    ref_p = Path(r['dir']) / 'df_trials_clean.pkl'
    if not ref_p.exists():
        continue
    _n_checked += 1
    mine = df_trials[df_trials.session == r.session]
    # Only compare where BOTH tables were built by the same task builder. A generic-builder session
    # defines a trial as collection-to-collection while the reference may be a spawn-batch table
    # (or, for the legacy JPAS_0231 file, an older overlapping-window schema entirely) -- comparing
    # those reports a difference of DEFINITION as though it were an error.
    if 'builder' in mine and (mine.builder == 'generic').any():
        print(f'{r.session}:  SKIPPED -- built by the generic builder, so the reference table is '
              f'a different trial definition, not a comparable one')
        continue
    ref = pd.read_pickle(ref_p)
    print(f'{r.session}:  mine {len(mine)} trials vs reference {len(ref)}')
    if len(mine) != len(ref):
        check('   same number of trials', False)
        continue
    check('   outcomes identical', (mine.outcome.values == ref.outcome.values).all())
    cols = [c for c in ['dur_s', 'path_efficiency', 'time_in_corner', 'mean_speed',
                        'heading_align'] if c in mine and c in ref]
    check('   geometry identical',
          all(np.allclose(mine[c].values.astype(float), ref[c].values.astype(float),
                          equal_nan=True) for c in cols), f'({", ".join(cols)})')
    if 'cluster_name' in ref and 'cluster_name' in mine:
        check('   cluster identical', (mine.cluster_name.values == ref.cluster_name.values).all())

if _n_checked == 0:
    print('No session in this directory has a df_trials_clean.pkl (none has been through the video '
          'pipeline), so there is nothing to cross-check here. This is expected for raw server data; '
          'the cross-check runs only where a full-pipeline reference table exists (e.g. flow_test).')

## C3 — SAVE

`.pkl` keeps everything, including the per-trial coordinate arrays and the `icons` lists. The
`.csv` of `df_sessions` is for eyeballing and sharing (it cannot carry the build stamp).

Each file is stamped with when and by what version it was built, so a stale copy is recognisable
rather than merely old.

In [ ]:
from pathlib import Path

# 1) canonical copy, next to the server data
OUT = MAIN_DIR / 'df_log'
OUT.mkdir(exist_ok=True)
for _f, _obj in [('df_sessions.pkl', df_sessions), ('df_trials.pkl', df_trials),
                 ('df_sessions.csv', df_sessions)]:
    bl.save(_obj, OUT / _f)

# 2) LOCAL copy, so notebook 2 (and you) can read it without the server mounted / re-downloading
LOCAL_OUT = Path('~/repo/session_pipeline_output').expanduser()
LOCAL_OUT.mkdir(parents=True, exist_ok=True)
for _f, _obj in [('df_sessions.pkl', df_sessions), ('df_trials.pkl', df_trials),
                 ('df_sessions.csv', df_sessions)]:
    bl.save(_obj, LOCAL_OUT / _f)

print(f'\nsaved to:\n  {OUT}\n  {LOCAL_OUT}   <- local copy, notebook 2 reads this one')
df_sessions.head()